# AWS In-Depth for Data Engineers
## S3, Glue, EMR, Redshift, Kinesis, Lambda & Production Architecture

**What you'll learn:**
- S3: storage classes, lifecycle policies, partitioning, performance tuning
- AWS Glue: ETL jobs, crawlers, catalog, Glue Studio, job bookmarks
- EMR: Spark on AWS, cluster types, spot instances, auto-scaling
- Redshift: architecture, distribution styles, sort keys, Spectrum, Serverless
- Kinesis: Data Streams, Firehose, Analytics -- real-time pipelines
- Lambda: serverless transforms, event-driven pipelines, limitations
- Step Functions: orchestration for complex workflows
- Lake Formation: fine-grained access control, data sharing
- MWAA: Managed Airflow on AWS
- MSK: Managed Kafka (Kafka on AWS)
- IAM: least-privilege policies for data pipelines
- Networking: VPC, endpoints, PrivateLink for data services
- Cost optimization: reserved capacity, spot, S3 tiering, right-sizing
- Reference architectures: batch, streaming, lakehouse on AWS
- Interview questions

**Estimated Time:** 10-12 hours

---

> AWS dominates the cloud market (~32% share). Most DE job postings
> list AWS services. This notebook covers every service you'll encounter.

---

---
# Section 1: Amazon S3 -- The Foundation of Everything

## S3 is Your Data Lake Storage

Every AWS data architecture starts with S3. It's:
- Unlimited storage (pay per GB)
- 99.999999999% durability (11 nines)
- 99.99% availability
- Object storage (not file system -- no rename, no append)

## S3 Key Concepts

| Concept | Description | DE Relevance |
|---------|-------------|--------------|
| **Bucket** | Top-level container (globally unique name) | One per environment/team |
| **Key** | Full path to object (prefix + filename) | Partition structure |
| **Prefix** | "Folder" path (S3 has no real folders) | Partition pruning |
| **Object** | File + metadata (up to 5TB per object) | Parquet files, logs, etc. |
| **Versioning** | Keep all versions of an object | Accidental delete recovery |
| **Lifecycle rules** | Auto-transition/delete objects by age | Cost optimization |

## S3 Storage Classes & Cost

| Class | Use Case | Cost (per GB/mo) | Retrieval |
|-------|----------|-------------------|-----------|
| **S3 Standard** | Frequently accessed (hot) | $0.023 | Instant |
| **S3 Standard-IA** | Infrequent access (30+ days) | $0.0125 | Instant, per-request fee |
| **S3 Intelligent-Tiering** | Unknown access pattern | $0.023-0.0036 | Auto-managed |
| **S3 Glacier Instant** | Archive, instant retrieval | $0.004 | Instant |
| **S3 Glacier Flexible** | Archive, hours retrieval | $0.0036 | 1-12 hours |
| **S3 Glacier Deep** | Long-term archive | $0.00099 | 12-48 hours |

## S3 Data Lake Partitioning (CRITICAL for performance)

```
# BAD: Flat structure (full scan for any query)
s3://data-lake/orders/orders_all.parquet  (500GB single file)

# GOOD: Partitioned by date (Spark/Athena only reads relevant partitions)
s3://data-lake/orders/
  year=2024/
    month=01/
      day=01/part-00000.parquet
      day=01/part-00001.parquet
      day=02/part-00000.parquet
    month=02/
      ...
  year=2023/
    ...

# Query: WHERE year=2024 AND month=06
# Only reads: s3://data-lake/orders/year=2024/month=06/  (tiny fraction!)
```

## S3 Performance Optimization

| Technique | What | When |
|-----------|------|------|
| **Prefix partitioning** | 3,500 PUT/s and 5,500 GET/s per prefix | Spread requests across prefixes |
| **Multipart upload** | Upload >100MB in parallel parts | Large file uploads |
| **S3 Transfer Acceleration** | CloudFront edge for uploads | Cross-region ingestion |
| **S3 Select** | Query subset of object with SQL | Read 1 column from CSV without downloading all |
| **Request Pays** | Requester pays transfer costs | Data sharing |
| **Byte-range fetches** | Download partial object | Parquet column reads (Spark does this!) |

In [ ]:
# S3 concepts and boto3 patterns
print("AWS S3 FOR DATA ENGINEERS")
print("=" * 60)

print("""
# COMMON BOTO3 PATTERNS (Python SDK for AWS):

import boto3
s3 = boto3.client('s3')

# 1. LIST OBJECTS (with pagination)
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket='data-lake', Prefix='orders/year=2024/'):
    for obj in page.get('Contents', []):
        print(f"  {obj['Key']} ({obj['Size'] / 1024 / 1024:.1f} MB)")

# 2. UPLOAD with multipart (for large files)
from boto3.s3.transfer import TransferConfig
config = TransferConfig(
    multipart_threshold=100 * 1024 * 1024,  # 100MB
    max_concurrency=10,
    multipart_chunksize=50 * 1024 * 1024,   # 50MB chunks
)
s3.upload_file('large_file.parquet', 'data-lake', 'output/file.parquet', Config=config)

# 3. LIFECYCLE POLICY (auto-tier old data)
s3.put_bucket_lifecycle_configuration(
    Bucket='data-lake',
    LifecycleConfiguration={
        'Rules': [{
            'ID': 'archive-old-data',
            'Filter': {'Prefix': 'raw/'},
            'Status': 'Enabled',
            'Transitions': [
                {'Days': 30, 'StorageClass': 'STANDARD_IA'},
                {'Days': 90, 'StorageClass': 'GLACIER'},
            ],
            'Expiration': {'Days': 365},  # Delete after 1 year
        }]
    }
)

# 4. S3 EVENT NOTIFICATION (trigger on new file)
# S3 → EventBridge → Lambda/Step Functions/Glue
# Pattern: New file lands → trigger ETL automatically

# 5. SPARK READING FROM S3 (most common DE operation)
df = spark.read.parquet("s3://data-lake/orders/year=2024/month=06/")
# Spark auto-discovers partitions and prunes based on filters
""")

# Cost estimation
print("S3 COST ESTIMATION:")
print(f"  {'Data Volume':<15} {'Storage/mo':<15} {'Class':<20}")
print(f"  {'-'*50}")
volumes = [(1, "TB"), (10, "TB"), (100, "TB"), (1, "PB")]
for size, unit in volumes:
    gb = size * 1024 if unit == "TB" else size * 1024 * 1024
    standard = gb * 0.023
    ia = gb * 0.0125
    glacier = gb * 0.004
    print(f"  {size} {unit:<12} ${standard:>10,.0f}     Standard")
    print(f"  {'':15} ${ia:>10,.0f}     Standard-IA")
    print(f"  {'':15} ${glacier:>10,.0f}     Glacier Instant")
    print()

---
# Section 2: AWS Glue -- Serverless ETL

## Glue Components

```
┌────────────────────────────────────────────────────────────┐
│                    AWS GLUE ECOSYSTEM                        │
│                                                              │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐     │
│  │  GLUE CATALOG│  │  GLUE ETL    │  │ GLUE CRAWLERS│     │
│  │  (Hive Meta- │  │  JOBS        │  │              │     │
│  │   store)     │  │  (Spark)     │  │ Auto-discover│     │
│  │              │  │              │  │ schema from  │     │
│  │ Tables,      │  │ PySpark or   │  │ S3/JDBC/DDB  │     │
│  │ Partitions,  │  │ Scala ETL    │  │              │     │
│  │ Schemas      │  │ Serverless   │  │ Populate     │     │
│  │              │  │              │  │ Catalog      │     │
│  └──────────────┘  └──────────────┘  └──────────────┘     │
│         │                  │                  │              │
│         └──────────────────┼──────────────────┘              │
│                            │                                 │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐     │
│  │ GLUE STUDIO  │  │ GLUE DATA   │  │ GLUE WORKFLOW│     │
│  │ (Visual ETL) │  │ QUALITY     │  │ (Orchestrate)│     │
│  │              │  │              │  │              │     │
│  │ Drag & drop  │  │ Rules engine │  │ Triggers,    │     │
│  │ job builder  │  │ for DQ checks│  │ crawlers,    │     │
│  │              │  │              │  │ jobs         │     │
│  └──────────────┘  └──────────────┘  └──────────────┘     │
└────────────────────────────────────────────────────────────┘
```

## Glue ETL Job Key Concepts

| Concept | Description |
|---------|-------------|
| **DPU** | Data Processing Unit (4 vCPU + 16GB RAM). You pay per DPU-hour |
| **Worker type** | G.1X (1 DPU/worker), G.2X (2 DPU), G.4X, G.8X, Z.2X (memory-optimized) |
| **Job Bookmark** | Tracks what's been processed (incremental ETL without reprocessing) |
| **Pushdown Predicate** | Filter at source (don't read all data into Spark) |
| **DynamicFrame** | Glue's extension of Spark DataFrame (handles schema ambiguity) |
| **Connection** | JDBC/network config for accessing RDS, Redshift, etc. |

## Glue Job Bookmark (Incremental Processing)

```python
# WITHOUT bookmark: processes ALL data every run (expensive!)
# WITH bookmark: only processes NEW data since last run

# Enable in job parameters:
# --job-bookmark-option: job-bookmark-enable

# Glue tracks:
#   - S3: last modification timestamp of processed files
#   - JDBC: last primary key value read
#   - Kafka: last offset consumed

# Equivalent to Delta Lake's "Auto Loader" but for Glue
```

---
# Section 3: Amazon EMR -- Spark at Scale on AWS

## EMR Cluster Architecture

```
EMR CLUSTER:
┌─────────────────────────────────────────────────────┐
│                                                       │
│  MASTER NODE (1):                                    │
│  - YARN ResourceManager                              │
│  - Spark Driver                                      │
│  - Hive Metastore                                    │
│  - Jupyter/Zeppelin                                  │
│                                                       │
│  CORE NODES (N): (always-on, store HDFS data)        │
│  - YARN NodeManager                                  │
│  - Spark Executors                                   │
│  - HDFS DataNodes                                    │
│  Recommended: On-Demand (data persistence)           │
│                                                       │
│  TASK NODES (M): (compute-only, no HDFS)             │
│  - Spark Executors only                              │
│  - Scale up/down based on workload                   │
│  Recommended: SPOT INSTANCES (70% cheaper!)          │
│                                                       │
└─────────────────────────────────────────────────────┘
```

## EMR vs Databricks vs Glue

| Feature | EMR | Databricks on AWS | Glue |
|---------|-----|------------------|------|
| **Management** | Self-managed clusters | Managed (Databricks) | Fully serverless |
| **Pricing model** | EC2 + EMR fee | DBU + EC2 | DPU-hours |
| **Startup time** | 5-10 minutes | 2-5 minutes | 30-60 seconds |
| **Auto-scaling** | Manual config or Managed Scaling | Native auto-scale | Automatic |
| **Notebook** | Jupyter, Zeppelin | Databricks Notebooks | Glue Studio |
| **Delta Lake** | Supported (install) | Native | Limited support |
| **Cost (typical)** | Lowest (with spot) | Medium | Highest per compute |
| **Best for** | Large batch, cost-sensitive | Interactive + production | Small-medium serverless ETL |
| **Spot instances** | Yes (task nodes) | Yes (workers) | N/A (serverless) |

---
# Section 4: Amazon Redshift -- Cloud Data Warehouse

## Redshift Architecture

```
LEADER NODE:
  - Receives queries from clients
  - Parses, optimizes, creates execution plan
  - Distributes work to compute nodes
  - Aggregates results

COMPUTE NODES (N):
  - Store data in columnar format
  - Execute query fragments in parallel
  - Each node has "slices" (parallel processing units)

  Node Types:
    ra3.xlplus:  4 vCPU, 32GB, managed storage (recommended)
    ra3.4xlarge: 12 vCPU, 96GB
    ra3.16xlarge: 48 vCPU, 384GB
    dc2.large:   2 vCPU, 15GB, 160GB SSD (legacy)
```

## Distribution Styles (CRITICAL for performance)

| Style | How Data is Distributed | Use When |
|-------|------------------------|----------|
| **KEY** | Rows with same key on same node | Large tables joined on this key |
| **EVEN** | Round-robin across nodes | No clear join key |
| **ALL** | Full copy on every node | Small dimension tables (<5M rows) |
| **AUTO** | Redshift decides (starts ALL, switches to KEY) | Default (recommended) |

```sql
-- Example: orders (large) distributed by customer_id
-- customers (small) distributed ALL (every node has full copy)
-- Join on customer_id = NO data movement!

CREATE TABLE orders (
    order_id BIGINT,
    customer_id INT,
    amount DECIMAL(10,2)
)
DISTSTYLE KEY
DISTKEY (customer_id)
SORTKEY (order_date);  -- Optimize for time-range queries

CREATE TABLE customers (
    customer_id INT,
    name VARCHAR(100)
)
DISTSTYLE ALL;  -- Small table, copied everywhere
```

## Redshift Spectrum (Query S3 Data Lake)

```sql
-- Create external schema pointing to Glue Catalog
CREATE EXTERNAL SCHEMA lake
FROM DATA CATALOG DATABASE 'data_lake_db'
IAM_ROLE 'arn:aws:iam::123456:role/RedshiftSpectrumRole';

-- Query S3 data directly (no loading needed!)
SELECT date, SUM(amount)
FROM lake.orders  -- This is in S3 (Parquet)!
WHERE year = 2024 AND month = 6
GROUP BY date;

-- Join local Redshift table with S3 data lake
SELECT c.name, SUM(o.amount)
FROM local_schema.customers c
JOIN lake.orders o ON c.id = o.customer_id
GROUP BY c.name;
```

## Redshift Serverless (2023+)

```
No cluster management! Just create a namespace + workgroup.
- Auto-scales compute (RPU: Redshift Processing Units)
- Pay per query (RPU-hours actually used)
- 128 RPU base = ~$12/hr when active, $0 when idle
- Best for: variable workloads, dev/test, small teams
```

---
# Section 5: Kinesis & Lambda -- Real-Time on AWS

## Kinesis Family

| Service | Purpose | Kafka Equivalent |
|---------|---------|-----------------|
| **Kinesis Data Streams** | Real-time streaming (shards) | Kafka topics/partitions |
| **Kinesis Data Firehose** | Auto-deliver to S3/Redshift (no code) | Kafka Connect S3 sink |
| **Kinesis Data Analytics** | SQL/Flink on streams | Kafka Streams / Flink |
| **MSK (Managed Kafka)** | Actual Kafka, managed | Kafka itself! |

## When to Use Kinesis vs MSK (Kafka)

| Factor | Kinesis | MSK (Kafka) |
|--------|---------|-------------|
| **Simplicity** | Simpler (AWS native) | More complex (full Kafka) |
| **Retention** | 24h-365 days | Unlimited |
| **Throughput** | 1 MB/s per shard | Much higher per partition |
| **Consumer model** | Pull (getRecords) | Pull (consumer groups) |
| **Replay** | Limited | Full (seek to any offset) |
| **Ecosystem** | AWS only | Multi-cloud, huge ecosystem |
| **Cost at scale** | Expensive | Cheaper for high throughput |
| **Best for** | AWS-native, simple streams | Serious streaming, multi-consumer |

## Lambda for Data Engineering

```
Common Lambda patterns for DE:

1. FILE-TRIGGERED ETL:
   S3 file uploaded → Lambda → validate + transform → write to processed/

2. API DATA COLLECTION:
   CloudWatch Event (every 5 min) → Lambda → call API → write to S3

3. LIGHTWEIGHT TRANSFORMS:
   Kinesis/Kafka → Lambda → enrich/filter → write to another stream

4. GLUE JOB ORCHESTRATION:
   Step Functions → Lambda (check conditions) → trigger Glue job

LIMITATIONS:
  - 15 minute max execution time
  - 10GB max memory (was 3GB, increased in 2022)
  - 250MB deployment package (512MB with layers)
  - Cold start: 100ms-2s (depends on runtime + VPC)
  - NOT for: large batch ETL, Spark jobs, long-running processes
```

---
# Section 6: AWS Data Platform Reference Architectures

## Batch Lakehouse Architecture

```
DATA SOURCES                     LAKEHOUSE                        CONSUMPTION
┌──────────┐                    ┌──────────────────────────┐     ┌──────────┐
│ RDS/Aurora│──CDC──>           │         S3 DATA LAKE      │     │ Redshift │
│ (OLTP)   │  (DMS) │         │                            │     │(warehouse)│
└──────────┘        │         │  Bronze    Silver    Gold   │     └──────────┘
                    │         │  (raw)    (clean)  (biz)   │          │
┌──────────┐        ▼         │    │         │        │    │     ┌──────────┐
│ SaaS APIs│──Lambda──>       │    ▼         ▼        ▼    │────>│ Athena   │
│(Salesforce│        │         │  ┌────┐   ┌────┐  ┌────┐  │     │(ad-hoc)  │
│ Stripe)  │        │         │  │Parq│──>│Parq│─>│Parq│  │     └──────────┘
└──────────┘        │         │  │uet │   │uet │  │uet │  │          │
                    ▼         │  └────┘   └────┘  └────┘  │     ┌──────────┐
┌──────────┐  ┌──────────┐   │                            │────>│QuickSight│
│ Files    │─>│   S3     │──>│  Glue Catalog (schemas)    │     │  (BI)    │
│(CSV,JSON)│  │ (landing)│   │  Lake Formation (security) │     └──────────┘
└──────────┘  └──────────┘   └──────────────────────────┘
                                       │
                              ┌────────────────────┐
                              │  ORCHESTRATION     │
                              │  MWAA (Airflow)    │
                              │  or Step Functions │
                              └────────────────────┘
```

## Streaming Architecture

```
SOURCES                   STREAMING              PROCESSING           SINKS
┌──────────┐            ┌──────────┐          ┌──────────┐        ┌──────────┐
│ App Events│──────────>│  MSK     │────────> │ Flink    │──────> │ S3 (lake)│
│ (Kafka   │            │ (Kafka)  │          │ (real-   │        │ Redshift │
│  producer)│            │          │          │  time)   │        │ DynamoDB │
└──────────┘            └──────────┘          └──────────┘        │ OpenSrch │
                             │                      │              └──────────┘
┌──────────┐            ┌──────────┐          ┌──────────┐
│ CDC      │──────────>│ Kinesis  │────────> │ Lambda   │──────> Alerts (SNS)
│(Debezium)│            │ Firehose │          │ (light   │
└──────────┘            │(auto S3) │          │  enrich) │
                        └──────────┘          └──────────┘
```

In [ ]:
# AWS services decision framework
print("AWS SERVICE DECISION FRAMEWORK")
print("=" * 60)

decisions = {
    "Storage": [
        ("Structured data, frequent queries", "Redshift or Athena+S3"),
        ("Semi-structured, data lake", "S3 (Parquet/Delta)"),
        ("Key-value, <10ms latency", "DynamoDB"),
        ("Relational, transactional", "RDS/Aurora"),
        ("Cache, sub-ms latency", "ElastiCache (Redis)"),
        ("Time-series", "Timestream"),
        ("Search", "OpenSearch"),
    ],
    "Processing": [
        ("Batch ETL, Spark", "EMR or Glue or Databricks"),
        ("Batch ETL, SQL-only", "Athena or Redshift"),
        ("Real-time, complex stateful", "MSK + Flink (Managed)"),
        ("Real-time, simple transforms", "Kinesis + Lambda"),
        ("Real-time, auto-deliver to S3", "Kinesis Firehose"),
        ("Small transforms, event-driven", "Lambda"),
        ("ML training", "SageMaker or EMR"),
    ],
    "Orchestration": [
        ("Complex DAGs, many dependencies", "MWAA (Airflow)"),
        ("Simple sequential workflows", "Step Functions"),
        ("Event-driven triggers", "EventBridge + Lambda"),
        ("Glue-specific pipelines", "Glue Workflows"),
    ],
    "Governance": [
        ("Fine-grained access to lake", "Lake Formation"),
        ("Schema registry", "Glue Catalog"),
        ("Data lineage", "DataZone or custom"),
        ("Compliance/audit", "CloudTrail + Config"),
    ],
}

for category, options in decisions.items():
    print(f"\n  {category}:")
    print(f"  {'-'*55}")
    for need, service in options:
        print(f"    {need:<40} -> {service}")

---
# Section 7: IAM & Security for Data Pipelines

## Principle of Least Privilege

```json
// BAD: Full admin access for a Glue job (overly permissive)
{
  "Effect": "Allow",
  "Action": "*",
  "Resource": "*"
}

// GOOD: Only what the Glue job needs
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Sid": "ReadSourceBucket",
      "Effect": "Allow",
      "Action": ["s3:GetObject", "s3:ListBucket"],
      "Resource": [
        "arn:aws:s3:::source-bucket",
        "arn:aws:s3:::source-bucket/raw/*"
      ]
    },
    {
      "Sid": "WriteTargetBucket",
      "Effect": "Allow",
      "Action": ["s3:PutObject", "s3:DeleteObject"],
      "Resource": "arn:aws:s3:::target-bucket/processed/*"
    },
    {
      "Sid": "GlueCatalogAccess",
      "Effect": "Allow",
      "Action": ["glue:GetTable", "glue:GetDatabase", "glue:UpdateTable"],
      "Resource": "arn:aws:glue:us-east-1:123456:catalog/*"
    }
  ]
}
```

## Security Best Practices for DE

| Practice | Implementation |
|----------|---------------|
| Encrypt at rest | S3: SSE-S3 or SSE-KMS (default now) |
| Encrypt in transit | TLS 1.2+ for all connections |
| Network isolation | VPC + private subnets for data services |
| VPC endpoints | S3, Glue, KMS endpoints (no public internet) |
| Cross-account access | Resource policies + assume role |
| Secrets management | Secrets Manager (not env vars!) |
| Audit logging | CloudTrail for all API calls |
| Data classification | Tags + Lake Formation permissions |

In [ ]:
# AWS cost optimization for DE
print("AWS COST OPTIMIZATION FOR DATA ENGINEERS")
print("=" * 60)

print("""
TOP 10 COST SAVINGS FOR DE ON AWS:

1. S3 LIFECYCLE POLICIES (save 60-80% on storage):
   - Raw data → Standard-IA after 30 days
   - Processed data → Glacier after 90 days
   - Delete temp/staging after 7 days

2. SPOT INSTANCES for EMR task nodes (save 70%):
   - Core nodes: On-Demand (data safety)
   - Task nodes: Spot (compute only, replaceable)
   - Use diversified instance types (m5.xlarge, m5a.xlarge, m4.xlarge)

3. RESERVED CAPACITY for always-on services (save 30-60%):
   - Redshift Reserved Nodes (1-3 year term)
   - RDS Reserved Instances
   - EMR core nodes (if running 24/7)

4. RIGHT-SIZE CLUSTERS:
   - Monitor CloudWatch: CPU utilization < 40%? Downsize!
   - Use auto-scaling for EMR/Redshift
   - Serverless for variable workloads (Redshift Serverless, Glue)

5. S3 SELECT / ATHENA PARTITION PRUNING:
   - Don't scan 1TB when you need 1GB
   - Partition by date/region, use columnar formats (Parquet)
   - Athena scans only cost: $5 per TB scanned

6. GLUE FLEX EXECUTION (save 34%):
   - Non-urgent ETL runs on Glue Flex (preemptible capacity)
   - Slightly slower start, but 34% cheaper per DPU

7. TURN OFF NON-PRODUCTION:
   - Dev/staging clusters: auto-stop after 6PM
   - Lambda + CloudWatch Events to start at 8AM
   - Savings: 65% of compute costs (run 12h instead of 24h)

8. DATA COMPRESSION:
   - Parquet + ZSTD = 5-10x smaller than CSV
   - Less storage cost + less scan cost + faster queries

9. REDSHIFT: CONCURRENCY SCALING vs bigger cluster:
   - Burst capacity for peak hours (first 1 hour/day free)
   - Cheaper than running bigger cluster 24/7

10. MONITOR WITH AWS COST EXPLORER + BUDGETS:
    - Set budget alerts (80%, 100%, 120% of expected)
    - Tag everything: team, project, environment
    - Review weekly, optimize monthly
""")

# Cost comparison
print("\nCOST COMPARISON (processing 1TB daily ETL):")
print(f"  {'Service':<25} {'Monthly Cost':<15} {'Notes'}")
print(f"  {'-'*55}")
estimates = [
    ("EMR (Spot, auto-scale)", "$800-1,500", "Cheapest, most control"),
    ("Glue (standard)", "$2,000-4,000", "Serverless, no cluster mgmt"),
    ("Glue (Flex)", "$1,300-2,600", "34% cheaper, slight delay"),
    ("Databricks (on AWS)", "$1,500-3,000", "Best developer experience"),
    ("Redshift Serverless", "$1,000-2,500", "SQL-only workloads"),
]
for service, cost, notes in estimates:
    print(f"  {service:<25} {cost:<15} {notes}")

In [ ]:
print("\nAWS INTERVIEW QUESTIONS FOR DATA ENGINEERS")
print("=" * 60)

questions = [
    {
        "q": "How would you design a data lake on AWS?",
        "a": "S3 for storage (Parquet format, partitioned by date). Glue Catalog for metadata (schemas, partitions). Lake Formation for access control. Medallion architecture: raw/ (Bronze), processed/ (Silver), analytics/ (Gold). Athena for ad-hoc queries, Redshift Spectrum for joining with warehouse. MWAA or Step Functions for orchestration."
    },
    {
        "q": "S3 consistency model -- what should you know?",
        "a": "Since Dec 2020, S3 provides strong read-after-write consistency for PUT/DELETE. New objects are immediately visible. Overwritten objects are immediately updated. This means no more eventual consistency issues for data pipelines! Before this, you had to add workarounds (S3Guard, retry logic)."
    },
    {
        "q": "How do you optimize Redshift query performance?",
        "a": "1) Choose right DISTKEY (co-locate joined data on same node). 2) Set SORTKEY on filter/range columns (zone maps skip blocks). 3) Use compression encodings (ANALYZE COMPRESSION). 4) Vacuum regularly (reclaim space from deletes). 5) Use Spectrum for cold data (keep hot data local). 6) Monitor with STL_QUERY and SVL_QUERY_REPORT."
    },
    {
        "q": "Kinesis vs MSK -- when do you choose each?",
        "a": "Kinesis: AWS-native, simpler, good for <50 MB/s per shard, tight AWS integration (Lambda, Firehose). MSK: when you need full Kafka (unlimited retention, consumer groups, replay, multi-cloud compatibility, higher throughput). Rule of thumb: Kinesis for simple AWS-only streams, MSK for serious streaming workloads."
    },
    {
        "q": "How do you handle data pipeline failures on AWS?",
        "a": "1) Idempotent operations (MERGE, overwrite partitions). 2) Dead-letter queues (SQS DLQ for Lambda/Kinesis failures). 3) Step Functions with retry + catch for complex workflows. 4) CloudWatch Alarms on Glue job failures. 5) SNS notifications to Slack/PagerDuty. 6) Automated rollback via CodePipeline."
    },
    {
        "q": "How do you secure data at rest and in transit on AWS?",
        "a": "At rest: S3 SSE-KMS (customer-managed keys), Redshift encryption, EBS encryption. In transit: TLS 1.2+ enforced via bucket policies (aws:SecureTransport). Network: VPC with private subnets, S3 VPC endpoints (no public internet). Access: IAM least-privilege, Lake Formation for fine-grained column/row security. Audit: CloudTrail for all API activity."
    },
]

for i, qa in enumerate(questions, 1):
    print(f"\nQ{i}: {qa['q']}")
    print(f"{'─'*60}")
    print(f"  {qa['a']}")

---
# Summary: AWS for DE Cheat Sheet

## Service Selection Quick Reference

| Need | Service | Key Feature |
|------|---------|-------------|
| Store anything | **S3** | Unlimited, $0.023/GB/mo |
| Query S3 with SQL | **Athena** | Serverless, $5/TB scanned |
| Full data warehouse | **Redshift** | Columnar, massively parallel |
| Serverless ETL | **Glue** | Spark-based, pay per DPU |
| Spark at scale | **EMR** | Full control, cheapest w/ spot |
| Real-time streaming | **MSK** | Managed Kafka |
| Simple streaming | **Kinesis** | AWS-native, Firehose for auto-S3 |
| Serverless compute | **Lambda** | Event-driven, 15min max |
| Orchestration | **MWAA / Step Functions** | Airflow or state machines |
| Data catalog | **Glue Catalog** | Hive-compatible metastore |
| Access control | **Lake Formation** | Column/row-level security |
| CDC from databases | **DMS** | Real-time replication |
| Search/analytics | **OpenSearch** | Elasticsearch managed |
| Cache | **ElastiCache** | Redis/Memcached managed |

## Cost Rules of Thumb

- S3 Standard: $23/TB/month
- Athena: $5/TB scanned (use partitions!)
- Glue: $0.44/DPU-hour (~$3.50/hr for 8-worker job)
- EMR: EC2 cost + 25% EMR fee (use spot for 70% savings)
- Redshift: $0.25/hr per node (ra3.xlplus) or serverless $0.375/RPU-hr
- Lambda: $0.20 per 1M requests + compute time

---
*AWS In-Depth for Data Engineers*